In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix)
import joblib
 
# ═══════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════
df = pd.read_csv("C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/data/processed/hp_landslide_dataset_v5_final.csv")
 
# ── Basic cleanup ──
df['soil_moisture'] = df['soil_moisture'].fillna(df['soil_moisture'].median())
df['ndvi']          = df['ndvi'].clip(-1, 1)
 
print("=== Dataset shape:", df.shape)
print("\nLabel counts:")
print(df['label'].value_counts())
 
# ═══════════════════════════════════════════════
# 2. SANITY CHECK — correlations should be balanced
#    rainfall_7day must NOT be near zero anymore
# ═══════════════════════════════════════════════
print("\n=== Correlation with label (rainfall must be > 0.15):")
corr = df.corr(numeric_only=True)['label'].sort_values(ascending=False)
print(corr.round(3))
 
rainfall_corr = abs(corr['rainfall_7day'])
assert rainfall_corr > 0.15, (
    f"❌ STOP: rainfall_7day correlation is only {rainfall_corr:.3f}. "
    "Dataset has the old co-correlation problem. Do NOT train on this."
)
print(f"\n✅ rainfall_7day correlation = {rainfall_corr:.3f} — dataset is good, proceeding...\n")
 
print("\nMean values per class:")
print(df.groupby('label')[['rainfall_7day','slope','elevation','soil_moisture','ndvi']].mean().round(2))
 
# ═══════════════════════════════════════════════
# 3. FEATURES & SPLIT
# ═══════════════════════════════════════════════
X = df[['rainfall_7day', 'slope', 'elevation', 'soil_moisture', 'ndvi']]
y = df['label']
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# ═══════════════════════════════════════════════
# 4. MODEL
# Increased min_samples_leaf to reduce overconfidence
# Reduced n_estimators slightly to avoid overfitting small dataset
# ═══════════════════════════════════════════════
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=200,       # reduced from 300 — dataset is only 245 rows
        max_depth=3,            # reduced from 4 — prevents overfit
        learning_rate=0.05,
        min_samples_leaf=12,    # increased from 8 — prevents overconfidence
        subsample=0.8,
        random_state=42
    ))
])
 
model.fit(X_train, y_train)
 
# ═══════════════════════════════════════════════
# 5. EVALUATE
# ═══════════════════════════════════════════════
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
 
print("=== Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("=== ROC-AUC: ", round(roc_auc_score(y_test, y_proba), 4))
print("\n=== Classification Report:\n", classification_report(y_test, y_pred))
print("=== Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
 
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"\n=== 5-fold CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
 
# ═══════════════════════════════════════════════
# 6. FEATURE IMPORTANCES
# rainfall_7day should now have meaningful importance
# ═══════════════════════════════════════════════
importances = model.named_steps['clf'].feature_importances_
print("\n=== Feature importances (rainfall must not be near zero):")
for feat, imp in sorted(zip(X.columns, importances), key=lambda x: -x[1]):
    print(f"  {feat:15s} {imp:.4f}  {'█' * int(imp*40)}")
 
rainfall_imp = dict(zip(X.columns, importances))['rainfall_7day']
if rainfall_imp < 0.05:
    print("\n⚠️  WARNING: rainfall_7day importance is still low. "
          "Check dataset balance before using this model.")
else:
    print(f"\n✅ rainfall_7day importance = {rainfall_imp:.4f} — model is learning rainfall properly.")
 
# ═══════════════════════════════════════════════
# 7. REAL-WORLD SANITY CHECKS
# These simulate what your live API will send
# All assertions must pass before saving model
# ═══════════════════════════════════════════════
def predict_case(rainfall, slope, elevation, soil_moisture=0.18, ndvi=0.45):
    features = pd.DataFrame(
        [[rainfall, slope, elevation, soil_moisture, ndvi]],
        columns=X.columns
    )
    prob = model.predict_proba(features)[0][1]
    risk = "HIGH" if prob >= 0.65 else "MEDIUM" if prob >= 0.40 else "LOW"
    return prob, risk
 
print("\n=== Real-world sanity checks ===")
tests = [
    # (label, rainfall, slope, elevation, expected_risk)
    # These 3 are the critical new tests — slope alone must NOT = HIGH
    ("Steep slope, DRY CONDITIONS → must be LOW/MEDIUM", 5,   40, 2200, ["LOW", "MEDIUM"]),
    ("Steep slope, DRY CONDITIONS → must be LOW/MEDIUM", 8,   35, 2050, ["LOW", "MEDIUM"]),
    ("Flat terrain, HEAVY RAIN   → must be LOW/MEDIUM", 120,  5, 1500, ["LOW", "MEDIUM"]),
    # These are the expected HIGH cases
    ("Shimla,  heavy rain + steep → HIGH",               80,  45, 2206, ["HIGH"]),
    ("Manali,  heavy rain + steep → HIGH",               90,  40, 2050, ["HIGH"]),
    ("Kullu,   heavy rain + steep → HIGH",               70,  35, 1220, ["HIGH"]),
    # These must be LOW
    ("Shimla,  dry + gentle slope → LOW",                 3,   9, 2206, ["LOW"]),
    ("Mandi,   dry + gentle slope → LOW",                 5,  10,  850, ["LOW"]),
]
 
all_passed = True
for label, rain, slp, elev, expected in tests:
    prob, risk = predict_case(rain, slp, elev)
    passed = risk in expected
    status = "✅" if passed else "❌"
    if not passed:
        all_passed = False
    print(f"  {status} {label}")
    print(f"     rainfall={rain}mm  slope={slp}°  → prob={prob:.3f}  risk={risk}\n")
 
# ═══════════════════════════════════════════════
# 8. SAVE — only if all checks pass
# ═══════════════════════════════════════════════
if all_passed:
    joblib.dump(model, "C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/model/landslide_model.pkl")
    print("✅ All checks passed — model saved!")
else:
    print("❌ Some sanity checks failed — model NOT saved.")
    print("   Review the failed cases above before saving.")
 








=== Dataset shape: (245, 7)

Label counts:
label
0    150
1     95
Name: count, dtype: int64

=== Correlation with label (rainfall must be > 0.15):
label            1.000
slope            0.324
rainfall_7day    0.239
soil_moisture    0.178
ndvi             0.091
elevation       -0.187
Name: label, dtype: float64

✅ rainfall_7day correlation = 0.239 — dataset is good, proceeding...


Mean values per class:
       rainfall_7day  slope  elevation  soil_moisture  ndvi
label                                                      
0              37.27  15.98    2371.11          17.16  0.34
1              72.56  26.60    1745.75          20.01  0.40
=== Accuracy: 0.8776
=== ROC-AUC:  0.9228

=== Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.90        30
           1       0.88      0.79      0.83        19

    accuracy                           0.88        49
   macro avg       0.88      0.86      0.87        49
weighted